In [1]:
%useLatestDescriptors
%use dataframe, kandy

// %use dataframe(1.0.0-Beta4n), kandy(0.8.3)

In [2]:
import util.Helpers

val helpers = Helpers()

In [35]:
fun queryByTimePeriodAndEntries(startYear: String, endYear: String, entries: Int) =
    """
SELECT
    pd.*
FROM
    powerlifting_data pd
        JOIN
    (
        SELECT
            meetname,
            date,
            weightclasskg,
            division,
            COUNT(*) AS lifter_count
        FROM
            powerlifting_data
        WHERE
            event = 'SBD' AND place != 'NS'
            AND date BETWEEN '$startYear-01-01' AND '$endYear-12-31'
            AND totalkg IS NOT NULL
        GROUP BY
            meetname, date, weightclasskg, division
        HAVING
            COUNT(*) >= $entries
    ) AS qualified_classes
    ON pd.meetname = qualified_classes.meetname
        AND pd.date = qualified_classes.date
        AND pd.weightclasskg = qualified_classes.weightclasskg
        AND pd.division = qualified_classes.division
WHERE
pd.event = 'SBD' AND pd.place != 'NS'
  AND pd.date BETWEEN '$startYear-01-01' AND '$endYear-12-31'
  AND pd.totalkg IS NOT NULL;
    """

In [4]:
fun queryByTimePeriod(startYear: String, endYear: String) =
    """
SELECT
    pd.*
FROM
    powerlifting_data pd
WHERE
  pd.date BETWEEN '$startYear-01-01' AND '$endYear-12-31'
    """

In [5]:
import java.util.Date
import org.jetbrains.kotlinx.dataframe.annotations.*

@DataSchema
interface LifterData {
    val place: String
    val meetname: String
    val date: Date
    val event: String
    val weightclasskg: String?
    val entries: Int
    val division: String?
    val squat1kg: Double?
    val squat2kg: Double?
    val squat3kg: Double?
    val bench1kg: Double?
    val bench2kg: Double?
    val bench3kg: Double?
    val deadlift1kg: Double?
    val deadlift2kg: Double?
    val deadlift3kg: Double?
    val best3squatkg: Double?
    val best3benchkg: Double?
    val best3deadliftkg: Double?
    val totalkg: Double?
    val successfulLifts: Int
}

In [36]:
val query = queryByTimePeriodAndEntries(startYear = "2025", endYear = "2025", entries = 3)
val data: DataFrame<*> = helpers.fetchResults(query)

In [38]:
val queryUnGrouped = queryByTimePeriod(startYear = "2025", endYear = "2025")
val dataUnGrouped: DataFrame<*> = helpers.fetchResults(queryUnGrouped)

In [59]:
// not needed, kotlin DF handles nulls fine
// val filledData = dataUnGrouped.cast<LifterData2>().fillNulls { weightclasskg }.with { "0" }

In [24]:
val entriesPerClass = dataUnGrouped.cast<LifterData>()
    .filter { event == "SBD" && place != "NS" && totalkg != null }
    .groupBy { meetname and date and weightclasskg and division }
        .aggregate {
        count().into("entries")
    }

In [25]:
val dataWithEntries = dataUnGrouped.cast<LifterData>().innerJoin(entriesPerClass)

dataWithEntries.count()

72688

In [39]:
val filtered =
    dataWithEntries.cast<LifterData>()
        .filter {
            event == "SBD" && place != "NS" && totalkg != null &&
            it.entries >= 3
        }

filtered.count()

48623

In [37]:
data.count()

47985

In [40]:
listOf(data.count(), filtered.count())

[47985, 48623]

In [41]:
data.count() - filtered.count()

-638

In [19]:
data.filter { weightclasskg == null }.count()

0

In [42]:
val nullWeightClasses = filtered.filter { weightclasskg == null }

nullWeightClasses.count()

638

In [43]:
nullWeightClasses.excludeJoin(data).count()

638

In [ ]:
result.describe()

In [ ]:
val columns = listOf(
    data.squat1kg, data.squat2kg, data.squat3kg,
    data.bench1kg, data.bench2kg, data.bench3kg,
    data.deadlift1kg, data.deadlift2kg, data.deadlift3kg
)

In [ ]:
fun addNumberOfSuccessfulLifts(data: DataFrame<LifterData>, firstPlaceOnly: Boolean = true): DataFrame<LifterData> {

    val filteredDf = if (firstPlaceOnly) data.filter { it.place == "1" } else data

    return filteredDf.add("successfulLifts") {
        columns.count { column -> column.getValue(it) > 0 }
    }
        .groupBy { successfulLifts }
        .aggregate {
            count().into("count")
        }
        .filter { successfulLifts >= 3 }
        .sortBy { successfulLifts }
}

In [ ]:
val winnersDataFrame = addNumberOfSuccessfulLifts(data.cast<LifterData>())

In [ ]:
winnersDataFrame

In [ ]:
plot(winnersDataFrame) {

    bars {
        x(successfulLifts)
        y(count)
    }
}
//    .save("distribution-of-winners.svg")

In [ ]:
val plotWinners = plot(winnersDataFrame) {

    bars {
        x(successfulLifts) 
        y(count) {
            axis.name = "Number of Winners"
            axis {
                breaks(listOf(500,1000,1500,2000,2500,3000,3500,4000,4500,5000), format = "d")
            }
        }
        fillColor = Color.hex("#fec92e")
        borderLine {
            color = Color.hex("#777777")
            width = 0.5
        }
    }
//    points {
//        x.constant(9)
//        y.constant(30)
//        symbol = Symbol.CIRCLE_FILLED
//        alpha = 0.0 // transparent
//    }
    layout {
        title = "Distribution of Winners by Successful Attempts"
        caption = "data: Open powerlfting meets 2023"
        size = 600 to 300
        xAxisLabel = "Successful Attempts"
        style {
            global {
                text {
                    fontFamily = FontFamily.custom("Helvetica Neue")
                }
                plotCanvas {
                    title {
                        hJust = 0.5
                        margin = Margin(10.0)
                        fontSize = 17.0
                    }
                    caption {
                        hJust = 1.0
                        margin = Margin(10.0, 0.0, 0.0, 0.0)
                    }
                    margin = Margin(0.0, 30.0, 0.0, 5.0)
                }
            }
        }
    }
}

plotWinners
// Alternatively you can save your chart as an svg or png
//plotWinners.save("distribution-of-winners-custom-formatting.svg")
//plotWinners.save("distribution-of-winners-custom-formatting.png")

In [ ]:
val allLiftersDataFrame = addNumberOfSuccessfulLifts(data.cast<LifterData>(), false)

In [ ]:
allLiftersDataFrame

In [ ]:
val dfRatioWinners =
    dataFrameOf(winnersDataFrame.rename("count").into("winners").columns() + allLiftersDataFrame.select("count").rename("count").into("allLifters").columns())
        .add("ratioWinners") {
            (column<Double>("winners").getValue(it) / column<Double>("allLifters").getValue(it)) * 100.0
        }

//column -> column.getValue(it) > 0

In [ ]:
dfRatioWinners

In [ ]:
kandyConfig.themeApplied = false

val plotRatioWinners = plot(dfRatioWinners) {

    bars {
        x(successfulLifts) 
        y(ratioWinners) {
            axis.name = "Percentage"
            axis {
                breaks(listOf(5.0,10.0,15.0,20.0,25.0), format = "{.0f}%")
            }
        }
        fillColor = Color.hex("#fec92e")
        borderLine {
            color = Color.hex("#777777")
            width = 0.5
        }
    }
    points {
        x.constant(9)
        y.constant(25)
        symbol = Symbol.CIRCLE_FILLED
        alpha = 0.0 // transparent
    }
    layout {
        title = "Percentage of First Places by Successful Lifts"
        subtitle = "At least 3 lifters in Weight Class"
        caption = "Data: Open IPF meets 2025"
        size = 600 to 400
        xAxisLabel = "Successful Attempts"
        style {
            global {
                text {
                    fontFamily = FontFamily.custom("Helvetica Neue")
                }
                plotCanvas {
                    title {
                        hJust = 0.5
                        margin = Margin(10.0)
                        fontSize = 14.0
                    }
                    subtitle {
                        hJust = 0.5
                        margin = Margin(5.0)
                        fontSize = 11.0
                    }
                    caption {
                        hJust = 1.0
                        margin = Margin(10.0, 0.0, 0.0, 0.0)
                    }
                    margin = Margin(5.0, 30.0, 20.0, 5.0)
                }
            }
        }
    }
}

plotRatioWinners
//plotRatioWinners.save("percetage-of-first-places.svg")

In [ ]:
val plotBunch = plotBunch {
    add(plotWinners, 0.0, 0.0, 600.0, 300.0)
    add(plotRatioWinners, 0.0, 300.0, 600.0, 300.0)
}

plotBunch
//plotBunch.save("plot-bunch-example.svg")

## Bonus analysis! Which was the most commonly missed lift?

In [ ]:
val missedLifts = column<Int>("missedLifts")

fun addWhichLiftsWereMissed(df: DataFrame<LifterData>): AnyFrame {

    return df.add(successfulLifts) {
        columns.count { lift -> it[lift].toInt() > 0 }
    }
        .add(missedLifts) {
            columns.count { lift -> it[lift].toInt() <= 0 }
        }
        .let { frame ->
            columns.fold(frame) { acc, lift ->
                acc.add("missed_${lift.name()}") {
                    if (it[lift].toInt() <= 0) 1 else 0
                }
            }
        }
        .filter { it[successfulLifts] == 8 }
        .groupBy { successfulLifts }
        .aggregate {
            columns
                .forEach { lift ->
                sum("missed_${lift.name()}") into "total_missed_${lift.name()}"
            }
        }
}

In [ ]:
val missedLiftsDataFrame = addWhichLiftsWereMissed(data.cast<LifterData>())


In [ ]:
missedLiftsDataFrame

In [ ]:
val labels = missedLiftsDataFrame.map { row ->
    listOf(
        "S1" to row["total_missed_squat1kg"],
        "S2" to row["total_missed_squat2kg"],
        "S3" to row["total_missed_squat3kg"],
        "B1" to row["total_missed_bench1kg"],
        "B2" to row["total_missed_bench2kg"],
        "B3" to row["total_missed_bench3kg"],
        "D1" to row["total_missed_deadlift1kg"],
        "D2" to row["total_missed_deadlift2kg"],
        "D3" to row["total_missed_deadlift3kg"]
    )
}
    .flatten()

val countDataFrame = dataFrameOf(
    missedLifts.name() to labels.map { it.first },
    count.name() to labels.map { it.second }
)

val plotMissedLifts = countDataFrame.plot {
    bars {
        x(missedLifts)
        y(count) { axis.name = "count of attempts missed" }
        fillColor = Color.hex("#fec92e")
        borderLine {
            color = Color.hex("#777777")
            width = 0.5
        }
    }
    layout {
        title = "Most Commonly Missed Lift - All lifters"
        caption = "data: Open powerlfting meets 2023"
        size = 600 to 400
        xAxisLabel = "Lift"
        style {
            global {
                text {
                    fontFamily = FontFamily.custom("Helvetica Neue")
                }
                plotCanvas {
                    title {
                        hJust = 0.5
                        margin = Margin(10.0)
                        fontSize = 17.0
                    }
                    subtitle {
                        hJust = 0.5
                        margin = Margin(5.0)
                    }
                    caption {
                        hJust = 1.0
                        margin = Margin(10.0, 0.0, 0.0, 0.0)
                    }
                    margin = Margin(5.0, 30.0, 20.0, 5.0)
                }
            }
        }
    }
}
plotMissedLifts
//plotMissedLifts.save("most-commonly-missed-lifts-2015-2024.svg")


Answer: 3rd attempt bench press, it's not even close!